Topic: Deduplicating Records Using ROW_NUMBER()

Purpose:
This script demonstrates how to remove duplicate records by using ROW_NUMBER()
with PARTITION BY and ORDER BY.

In [0]:
%sql

DROP TABLE IF EXISTS customer_profiles;

CREATE TABLE customer_profiles (
    record_id INT PRIMARY KEY,
    customer_id INT NOT NULL,
    customer_name VARCHAR(100) NOT NULL,
    email VARCHAR(100),
    city VARCHAR(50),
    updated_at DATE NOT NULL
);

INSERT INTO customer_profiles (
    record_id,
    customer_id,
    customer_name,
    email,
    city,
    updated_at
) VALUES
(1, 101, 'Avishek Bhandari', 'avishek_old@example.com', 'Detroit', '2026-01-01'),
(2, 101, 'Avishek Bhandari', 'avishek_new@example.com', 'Detroit', '2026-01-05'),
(3, 102, 'John Smith', 'john@example.com', 'Chicago', '2026-01-02'),
(4, 102, 'John Smith', 'john.updated@example.com', 'Chicago', '2026-01-04'),
(5, 103, 'Maria Lopez', 'maria@example.com', 'Dallas', '2026-01-03'),
(6, 104, 'Sara Khan', 'sara@example.com', 'New York', '2026-01-04'),
(7, 104, 'Sara Khan', 'sara.latest@example.com', 'New York', '2026-01-04');



num_affected_rows,num_inserted_rows
7,7


In [0]:
%sql

-- Define a Common Table Expression (CTE) named 'ranked_customers' 
-- This temporary result calculates rankings for each row
WITH ranked_customers AS (
    SELECT 
        record_id,
        customer_id,
        customer_name,
        email,
        city,
        updated_at,
        -- ROW_NUMBER assigns a unique sequential integer to rows.
        -- PARTITION BY groups the data by customer_id (resets counting for each customer).
        -- ORDER BY sorts the group by the newest date first, using record_id as a fallback tie-breaker.
        ROW_NUMBER() OVER (
            PARTITION BY customer_id 
            ORDER BY updated_at DESC, record_id DESC
        ) AS row_num 
    FROM customer_profiles
)
-- Main query to select data from the temporary CTE
SELECT 
    record_id,
    customer_id,
    customer_name,
    email,
    city,
    updated_at
FROM ranked_customers
-- Filter for row_num = 1 to return only the single newest version of each customer's profile
WHERE row_num = 1
-- Sort the final deduplicated list by customer ID order
ORDER BY customer_id;

record_id,customer_id,customer_name,email,city,updated_at
2,101,Avishek Bhandari,avishek_new@example.com,Detroit,2026-01-05
4,102,John Smith,john.updated@example.com,Chicago,2026-01-04
5,103,Maria Lopez,maria@example.com,Dallas,2026-01-03
7,104,Sara Khan,sara.latest@example.com,New York,2026-01-04
